# Module 5: Fashion Image Exploratory Analysis
## Visual Analytics, K-Means Color Extraction & Color Harmony Matching

This notebook covers:
1. Multi-dimensional category and demographic analysis (Gender, Usage, Season).
2. K-Means pixel clustering for dominant color extraction with studio background filtering.
3. Color space conversion (RGB to HSV) and color wheel distribution.
4. Fashion color harmony rules (Complementary, Monochromatic, Analogous, Neutrals).
5. Coordinated visual style archetypes (Casual, Formal, Sports).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns

from src.eda_analysis import ColorExtractor, DatasetAnalyzer
from src.config import CLEANED_METADATA_CSV, EDA_FIGURES_DIR

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

### 1. Load Dataset & Overview

In [ ]:
df = pd.read_csv(CLEANED_METADATA_CSV)
print(f"Total items: {len(df):,}")
df.head()

### 2. Category & Demographic Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Category counts
cat_counts = df["canonical_category"].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, ax=axes[0], palette="viridis", hue=cat_counts.index, legend=False)
axes[0].set_title("Garment Catalog by Category", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Item Count")

# Outfit parts
part_counts = df["outfit_part"].value_counts()
sns.barplot(x=part_counts.values, y=part_counts.index, ax=axes[1], palette="mako", hue=part_counts.index, legend=False)
axes[1].set_title("Outfit Part Distribution (Top, Bottom, Shoes, Accessories)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Item Count")

plt.tight_layout()
plt.show()

### 3. Gender $\times$ Category Cross-Tabulation Heatmap

In [ ]:
top_genders = df["gender"].value_counts().head(4).index
crosstab_gender = pd.crosstab(df[df["gender"].isin(top_genders)]["gender"], df["canonical_category"])

plt.figure(figsize=(14, 5))
sns.heatmap(crosstab_gender, annot=True, fmt="d", cmap="YlGnBu", cbar=True)
plt.title("Demographic Distribution Across Clothing Types", fontsize=14, fontweight="bold")
plt.show()

### 4. Dominant Color Extraction via K-Means Clustering
Extracting top 3 color swatches per garment while automatically removing white/gray studio background pixels.

In [ ]:
sample_items = df[df["canonical_category"].isin(["T-Shirt", "Shirt", "Dress", "Shoes"])].sample(4, random_state=42)

fig, axes = plt.subplots(4, 2, figsize=(10, 12), gridspec_kw={"width_ratios": [1, 2]})

for idx, (_, row) in enumerate(sample_items.iterrows()):
    img_path = row["image_path"]
    img = Image.open(img_path)
    axes[idx, 0].imshow(img)
    axes[idx, 0].set_title(f"{row['productDisplayName'][:24]}...\n({row['canonical_category']})", fontsize=10, fontweight="bold")
    axes[idx, 0].axis("off")
    
    # Extract top 3 dominant colors
    palette = ColorExtractor.extract_dominant_colors(img_path, k=3)
    
    # Draw color swatch bars
    bar_y = 0
    for color_info in palette:
        rgb = [c / 255.0 for c in color_info["rgb"]]
        pct = color_info["percentage"]
        axes[idx, 1].barh(bar_y, pct, color=rgb, edgecolor="black", height=0.6)
        axes[idx, 1].text(pct + 0.02, bar_y, f"{color_info['hex']} ({pct*100:.1f}%) | {'Neutral' if color_info['is_neutral'] else 'Chromatic'}", va="center", fontsize=9, fontweight="bold")
        bar_y += 1
        
    axes[idx, 1].set_xlim(0, 1.25)
    axes[idx, 1].set_yticks([])
    axes[idx, 1].set_xlabel("Cluster Proportion")
    axes[idx, 1].set_title(f"Extracted Dominant Color Swatches", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

### 5. Fashion Color Harmony Rules
Demonstrating algorithmic pairing logic used in our outfit recommendation engine.

In [ ]:
pairs = [
    (("Black T-Shirt", (20, 20, 20)), ("Blue Denim Jeans", (45, 85, 140))),
    (("Navy Blue Blazer", (20, 35, 75)), ("White Chinos", (245, 245, 245))),
    (("Cobalt Blue Dress", (20, 80, 220)), ("Amber Orange Bag", (230, 130, 20))),
    (("Forest Green Sweater", (35, 90, 45)), ("Burgundy Scarf", (120, 20, 40))),
]

print(f"{'Item 1':<25} | {'Item 2':<25} | {'Calculated Harmony Rule'}")
print("-" * 78)
for (name1, c1), (name2, c2) in pairs:
    harmony = ColorExtractor.compute_color_harmony_type(c1, c2)
    print(f"{name1:<25} | {name2:<25} | {harmony.upper()}")